# Module 1 — K-12 Diagnostic Assessment Generator

In [ ]:
!pip install google-generativeai -q

In [ ]:
import google.generativeai as genai
import json

GEMINI_API_KEY = ""
MODEL_NAME    = "gemini-2.0-flash"

genai.configure(api_key=GEMINI_API_KEY)

In [ ]:
def build_assessment_prompt(subject, topic, grade_level, num_questions,
                             curriculum='', language='English'):
    easy   = round(num_questions * 0.4)
    medium = round(num_questions * 0.4)
    hard   = num_questions - easy - medium

    return f"""You are an expert educational assessment designer specializing in K-12 education,
curriculum alignment, cognitive diagnostics, and learning science.

Your responsibility is to accurately diagnose a student's conceptual understanding.

Subject: {subject}
Topic: {topic}
Target Grade: {grade_level}
Curriculum: {curriculum or 'General'}
Language: {language}
Number of Questions: {num_questions}

Difficulty Distribution:
  Easy:   {easy} questions  (40%)
  Medium: {medium} questions (40%)
  Hard:   {hard} questions  (20%)

RULES:
- Each question assesses exactly ONE micro-skill.
- Include common misconceptions as distractors.
- Questions must progress Easy → Medium → Hard.
- Use age-appropriate language for {grade_level}.
- Return ONLY valid JSON. No markdown, no explanations.

Every question must include: id, micro_skill, concept_category, difficulty,
blooms_level, question, options (A/B/C/D), correct_option, misconception_tested,
explanation, distractor_analysis (A/B/C/D), estimated_time_seconds,
grade_alignment_confidence.

Return this exact structure:
{{
  "subject": "{subject}",
  "topic": "{topic}",
  "grade": "{grade_level}",
  "curriculum": "{curriculum or 'General'}",
  "language": "{language}",
  "assessment_metadata": {{
    "difficulty_distribution": {{"Easy": "{easy}/{num_questions}", "Medium": "{medium}/{num_questions}", "Hard": "{hard}/{num_questions}"}},
    "estimated_completion_minutes": 0
  }},
  "questions": [
    {{
      "id": 1,
      "micro_skill": "",
      "concept_category": "",
      "difficulty": "Easy",
      "blooms_level": "Remember",
      "question": "",
      "options": {{"A": "", "B": "", "C": "", "D": ""}},
      "correct_option": "A",
      "misconception_tested": "",
      "explanation": "",
      "distractor_analysis": {{"A": "", "B": "", "C": "", "D": ""}},
      "estimated_time_seconds": 45,
      "grade_alignment_confidence": 0.98
    }}
  ]
}}

Generate exactly {num_questions} questions: {easy} Easy, {medium} Medium, {hard} Hard."""

In [ ]:
def generate_assessment(subject, topic, grade_level, num_questions,
                         curriculum='', language='English'):
    model = genai.GenerativeModel(
        model_name=MODEL_NAME,
        generation_config={
            'response_mime_type': 'application/json',
            'temperature': 0.3,
            'max_output_tokens': 8192,
        }
    )
    prompt = build_assessment_prompt(
        subject, topic, grade_level, num_questions, curriculum, language
    )
    response = model.generate_content(prompt)
    return json.loads(response.text)

In [ ]:
assessment = generate_assessment(
    subject       = 'Mathematics',
    topic         = 'Fractions',
    grade_level   = 'Grade 5',
    num_questions = 10,
    curriculum    = 'CBSE',
    language      = 'English',
)

print(assessment['topic'])
print(assessment['grade'])
print(len(assessment['questions']))

In [ ]:
for q in assessment['questions']:
    print(q['id'], q['difficulty'], q['blooms_level'], q['micro_skill'])
    print(q['question'])
    for opt, text in q['options'].items():
        print(opt, text)
    print(q['explanation'])
    print(q['misconception_tested'])

In [ ]:
filename = f"assessment_{assessment['topic'].lower().replace(' ','_')}.json"
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(assessment, f, indent=2, ensure_ascii=False)